# Phase 4 — Synthetic recovery environment

This notebook audits a deterministic experimental world. It does not claim that these probabilities, recovery times, intervention costs, or scenarios were observed from Razorpay customers.

The simulator's hidden probability and latent variation are ground-truth mechanisms for generating outcomes. They must never be passed to a trained model.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        path = candidate / "ml" / "data" / "processed" / "intervention_outcomes.csv"
        if path.exists():
            return candidate
    raise FileNotFoundError("Run `python -m ml.src.simulate_recovery` first")


ROOT = find_repo_root()
PROCESSED = ROOT / "ml" / "data" / "processed"
outcomes = pd.read_csv(PROCESSED / "intervention_outcomes.csv")
features = pd.read_csv(PROCESSED / "failed_payment_features.csv")
summary = json.loads((PROCESSED / "recovery_simulation_summary.json").read_text())

print("Outcomes:", outcomes.shape)
print("Failed payments:", outcomes["payment_id"].nunique())
summary

In [ ]:
assert len(outcomes) == 49_504
assert outcomes["payment_id"].nunique() == 12_376
assert not outcomes.duplicated(["payment_id", "intervention"]).any()
assert outcomes.groupby("payment_id")["intervention"].nunique().eq(4).all()
assert outcomes.isna().sum().sum() == 0
assert outcomes.loc[outcomes["recovered"].eq(0), "amount_recovered"].eq(0).all()
assert outcomes.loc[outcomes["policy_allowed"].eq(0), "recovered"].eq(0).all()
print("Potential-outcome coverage and money invariants are valid.")

In [ ]:
intervention_metrics = outcomes.groupby("intervention").agg(
    rows=("recovered", "size"),
    policy_allowed_rate=("policy_allowed", "mean"),
    mean_probability=("simulated_recovery_probability", "mean"),
    recovery_rate=("recovered", "mean"),
    recovered_amount=("amount_recovered", "sum"),
    median_recovery_hours=("time_to_recovery_hours", lambda values: values[values.gt(0)].median()),
)
intervention_metrics

In [ ]:
allowed = outcomes.loc[outcomes["policy_allowed"].eq(1)]
best = (
    allowed.sort_values(
        ["payment_id", "simulated_recovery_probability", "intervention"],
        ascending=[True, False, True],
    )
    .drop_duplicates("payment_id")
)
print(best["intervention"].value_counts())
best["intervention"].value_counts().plot.bar(title="Best allowed intervention by payment")

In [ ]:
calibration = allowed.assign(
    probability_bucket=pd.cut(
        allowed["simulated_recovery_probability"],
        bins=np.linspace(0, 1, 11),
        include_lowest=True,
    )
).groupby("probability_bucket", observed=True).agg(
    rows=("recovered", "size"),
    mean_probability=("simulated_recovery_probability", "mean"),
    actual_recovery_rate=("recovered", "mean"),
)
display(calibration)
calibration[["mean_probability", "actual_recovery_rate"]].plot(
    marker="o",
    title="Simulator probability calibration",
    ylim=(0, 1),
)

In [ ]:
audit = allowed.merge(
    features[[
        "transaction_id",
        "has_prior_history",
        "historical_success_rate",
        "amount_vs_previous_avg",
    ]],
    left_on="payment_id",
    right_on="transaction_id",
    validate="many_to_one",
)
audit["success_rate_bucket"] = pd.cut(
    audit["historical_success_rate"],
    bins=[-0.01, 0.4, 0.6, 0.8, 0.9, 1.0],
)
audit.groupby("success_rate_bucket", observed=True).agg(
    rows=("recovered", "size"),
    mean_probability=("simulated_recovery_probability", "mean"),
    recovery_rate=("recovered", "mean"),
)

In [ ]:
audit["amount_ratio_bucket"] = pd.cut(
    audit["amount_vs_previous_avg"],
    bins=[-0.01, 0.5, 1, 2, 5, 10, np.inf],
)
amount_sensitivity = audit.groupby("amount_ratio_bucket", observed=True).agg(
    rows=("recovered", "size"),
    mean_probability=("simulated_recovery_probability", "mean"),
    recovery_rate=("recovered", "mean"),
)
print("Raw maximum ratio:", features["amount_vs_previous_avg"].max())
print("Simulator transform is capped before log1p at 10.")
amount_sensitivity

In [ ]:
scenario_intervention = outcomes.pivot_table(
    index="synthetic_failure_scenario",
    columns="intervention",
    values="simulated_recovery_probability",
    aggfunc="mean",
)
scenario_intervention

In [ ]:
fraud = outcomes.merge(
    features[["transaction_id", "fraud_flag"]],
    left_on="payment_id",
    right_on="transaction_id",
    validate="many_to_one",
).query("fraud_flag == 1")

fraud.groupby("intervention").agg(
    rows=("payment_id", "size"),
    allowed=("policy_allowed", "sum"),
    recoveries=("recovered", "sum"),
)

## Boundaries for Phase 5

- `simulated_recovery_probability` is available only to audit the synthetic environment. It is forbidden as a model feature.
- Hidden responsiveness, payment shocks, and customer/intervention preferences are intentionally absent from the output.
- `synthetic_failure_scenario` is generated environment metadata, not a Kaggle or Razorpay observation.
- These are four potential outcomes per payment. A real historical-policy dataset must expose only one chosen intervention and one observed outcome.
- Fraud blocking is a deterministic policy constraint, not evidence that fraud predicts recoverability.
- Recovery times and zero costs are versioned experimental assumptions, not production claims.